In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from math import ceil
import matplotlib.animation as animation
from typing import Tuple, Optional, Union
from psim_kernels import neigh_count, neigh_list

In [ ]:
class PSimSystem:
    particle_id : torch.Tensor
    pos : torch.Tensor
    force : torch.Tensor
    shape : torch.Tensor
    cell_id : torch.Tensor
    cell_offsets : torch.Tensor
    neigh_counts : torch.Tensor
    neigh_count_offsets : torch.Tensor
    neigh_list : torch.Tensor
    
    
    def __init__(self, num_cells_per_dim : Union[int, Tuple[int, int, int]], cell_size : Union[float, Tuple[float, float, float]], device='cuda'):
        if isinstance(num_cells_per_dim, Tuple):
            self.num_cells_per_dim = num_cells_per_dim
            self.num_cells = self.num_cells_per_dim[0] * self.num_cells_per_dim[1] * self.num_cells_per_dim[2]
        elif isinstance(num_cells_per_dim, int):
            self.num_cells_per_dim = (num_cells_per_dim, num_cells_per_dim, num_cells_per_dim)
            self.num_cells = num_cells_per_dim ** 3
        else:
            raise ValueError("num_cells_per_dim must be either an integer or a tuple of three integers")
        self.num_cells_per_dim_tensor = torch.tensor(self.num_cells_per_dim, dtype=torch.long, device=device)
        if isinstance(cell_size, Tuple):
            self.cell_size = cell_size
        elif isinstance(cell_size, float):
            self.cell_size = (cell_size, cell_size, cell_size)
        else:
            raise ValueError("cell_size must be either a float or a tuple of three floats")
        self.cell_size_vec = torch.tensor(self.cell_size, dtype=torch.float32, device=device)
        self.box_size = (self.num_cells_per_dim[0] * self.cell_size[0], 
                            self.num_cells_per_dim[1] * self.cell_size[1], 
                            self.num_cells_per_dim[2] * self.cell_size[2])
        self.box_size_vec = torch.tensor(self.box_size, dtype=torch.float32, device=device)
        self.cell_offsets = torch.empty(self.num_cells + 1, dtype=torch.long, device=device)
        self.device = device

        self.particle_id = None
        self.pos = None
        self.force = None
        self.shape = None
        self.cell_id = None
        self.neigh_counts = None
        self.neigh_count_offsets = None
        self.neigh_list = None

    def lattice_init(self, num_particles : int, spacing : float):
        n_lattice = int(ceil(num_particles**(1/3)))
        indexing = torch.cartesian_prod(*[torch.arange(n_lattice) for _ in range(3)]).to(self.device)
        indexing = indexing[torch.randperm(indexing.size(0), device=self.device)[:num_particles]]
        self.pos = (indexing + 0.5) * spacing
        self.force = torch.zeros_like(self.pos)
        self.shape = torch.ones((num_particles, 3, 3), dtype=torch.long, device=self.device)
        self.particle_id = torch.arange(num_particles, dtype=torch.long, device=self.device)
        self.neigh_counts = torch.empty(num_particles, dtype=torch.long, device=self.device)
        self.neigh_count_offsets = torch.empty(num_particles + 1, dtype=torch.long, device=self.device)
    def update_neighbor_list(self, cutoff_skin : float):
        cell_coords = torch.floor(self.pos / self.cell_size_vec).long() % self.num_cells_per_dim_tensor
        nx, ny, nz = self.num_cells_per_dim
        cell_id = cell_coords[:, 0] + (cell_coords[:, 1] + cell_coords[:, 2] * ny)*nx
        cell_id_sorted, perm = torch.sort(cell_id) 
        self.cell_id = cell_id_sorted
        self.particle_id = self.particle_id[perm].contiguous()
        self.pos = self.pos[perm].contiguous()
        self.shape = self.shape[perm].contiguous()
        counts = torch.bincount(self.cell_id, minlength=self.num_cells)
        self.cell_offsets[0] = 0
        self.cell_offsets[1:] = torch.cumsum(counts, dim=0)
        neigh_count(pos=self.pos,
                     cell_ids=self.cell_id, 
                     cell_offsets=self.cell_offsets, 
                     num_cells_per_dim=self.num_cells_per_dim,
                     box_size=self.box_size,
                     neigh_counts=self.neigh_counts,
                     neigh_count_offsets=self.neigh_count_offsets,
                     cutoff_skin=cutoff_skin,
                     use_pbc=True,
                     ring=1
                     )
        self.neigh_count_offsets[0] = 0
        self.neigh_count_offsets[1:] = torch.cumsum(self.neigh_counts, dim=0)
        total_neighbors = self.neigh_count_offsets[-1]
        if self.neigh_list is None or self.neigh_list.shape[0] < total_neighbors:
            self.neigh_list = torch.empty(total_neighbors, dtype=torch.long, device=self.device)    
        neigh_list(pos=self.pos, cell_ids=self.cell_id, cell_offsets=self.cell_offsets, neigh_counts=self.neigh_counts, neigh_count_offsets=self.neigh_count_offsets, neigh_list=self.neigh_list)

In [3]:
psys = PSimSystem(num_cells_per_dim=10, cell_size=1.0)
psys.lattice_init(num_particles=1000, spacing=0.9)

In [4]:
psys.update_neighbor_list()

TypeError: neigh_count() missing 5 required positional arguments: 'num_cells_per_dim', 'box_size', 'cutoff_skin', 'neigh_count_offsets', and 'neigh_list'